In [ ]:
%pip install pandas numpy matplotlib statsmodel seaborn

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import statsmodels.api as sm
import seaborn as sns

# Linear Regression
## 4.3        (Linear Regression)  In this assignment, you will analyze linear regression models on different categories of data about average households in the United States.

## Gather and Prepare Data

###  Load zeta dataset, and remove all  meanhouseholdincome duplicates (or more formally only females records should be in the dataset) that does not have any duplicate rows of data (done in homework 3.2).

In [ ]:
raw = pd.read_csv("resources/4-lab/zeta.csv").drop(columns=['Unnamed: 0'])
raw = raw[raw['sex'] == 'F']
raw = raw.drop_duplicates(subset=['meanhouseholdincome'], keep='first')

## ПРОВЕРЯЮ, ЕСТЬ ЛИ ДУБЛИКАТЫ ИЛИ МУЖЧИНЫ
if (raw['meanhouseholdincome'].value_counts().max() != 1 or len(raw[raw['sex'] == 'M'])):
    print("DATASET IS BAD")
    
raw

### Remove the columns zcta and sex from the imported table.

In [ ]:
raw = raw.drop(columns=['zcta', 'sex'])
raw

### Remove outliers by creating subsets of the original data so that: 

- ### 8 < meaneducation < 18

- ### 10,000 < meanhouseholdincome < 200,000

- ### 0 < meanemployment < 3

- ### 20 < meanage < 60

In [ ]:
raw = raw[
    raw['meaneducation'].between(8, 18, inclusive='neither') &
    raw['meanhouseholdincome'].between(10_000, 200_000, inclusive='neither') &
    raw['meanemployment'].between(0, 3, inclusive='neither') &
    raw['meanage'].between(20, 60, inclusive='neither')
]

raw

### Create a variable called log_income = log10(meanhouseholdincome).

In [ ]:
raw['log_income'] = np.log10(raw['meanhouseholdincome'])
raw = raw.drop(columns=['meanhouseholdincome'])
raw

###  Rename the columns meanage, meaneducation, and meanemployment as age, education, and employment, respectively.

In [ ]:
raw = raw.rename(
    columns={
        'meanage' : 'age',
        'meaneducation' : 'education',
        'meanemployment' : 'employment'
    }
)

raw

### После очистки данных перенесем raw dataset в df и обновим индексы

In [ ]:
df = raw.reset_index(drop=True)
del(raw)
df

# Linear Regression Analysis

### a.      We will be analyzing this data with income as the dependent variable and the other columns as independent variables.  Create a scatter plot showing the effect age has on log_income and paste it here.  Do you see any linear relationship between the two variables?

Нет, линейной зависимости не наблюдается

In [ ]:
plt.figure(figsize=(8, 5))

plt.scatter(
    x = df['age'], 
    y = df['log_income'], 
    alpha=0.5,
)
plt.title('Effect age has on log_income')
plt.xlabel('Age')
plt.ylabel('Income')
plt.grid(True)

plt.show()

### b.      Create a linear regression model between log_income and age.  What is the interpretation of the t-value? What kind of t-value would indicate a significant coefficient?

In [ ]:
X = sm.add_constant(df['age'])
Y = df['log_income']

model = sm.OLS(Y, X).fit()

- ```t-value``` - отношение расчетного коэффициента и стандартной ошибки
  $$\text{Стандартная ошибка}(b_i) = \sigma_i^2 (X^T X)^{-1}$$

t-value хоть и очень высокое, у нас всего один фактор, отчего от него модель и очень сильно зависит, нельзя точно судить, нужен он нам или нет, но следует "взять на карандашик"

In [ ]:
print(model.summary())

### c.       What is the interpretation of the R-squared value?  What kind of R-squared value would indicate a good fit?

- ```R-квадрат```

$$R^2 = 1 - \frac{\sum (y_i - \hat{y}_i)^2}{\sum (y_i - \bar{y})^2}$$

R-квадрат всего 0.01, что является очень низким значением
В идеале. он должен быть хотя бы больше 0.5
Но для статистически сложных задач являются достаточными и значения выше 0.2 

### d.      What is the interpretation of the F-statistic?  What kind of F-statistic indicates a strong linear regression model?

- ```F-statistic``` - критерий Фишера (Отношение дисперсии к остаточной дисперсии)

  $$F = \frac{R^2 / k}{(1 - R^2) / (N - k - 1)}, \ - дополнительная \ формула$$

- Остаточная дисперсия 
$$\text{S}_{\text{ост}} = \frac{\sum_{i=1}^{N} (y_i - \hat{y}_i)^2}{N - k - 1}$$

- Объясненная дисперсия
$$\text{S}_{\text{объясн}} = \frac{\sum_{i=1}^{N} (\hat{y}_i - \bar{y})^2}{k}$$
  
  
- k - количество факторов
- N - количество данных


По таблице для  $\alpha = 0.05$ критическое табличное значение равняется 3.84
Расчетное F больше табличного => нулевая гипотеза отвергается

In [ ]:
print(f"Критерий Фишера равен {float(model.fvalue):.3f}")

### e.      View a detailed summary of the previous model.  What is the R-squared value?  Does this suggest that the model is a good fit? Why?

R-квадрат равен 0.012, из чего следует, что модель неадекватная (bad fit)

### f.        Create a scatter plot showing the effect education has on log_income.  Do you see any linear relationship between the two variables?
Да, здесь видна линейная зависимость между фактором и целевой переменной

In [ ]:
plt.figure(figsize=(8, 5))

plt.scatter(
    x = df['education'], 
    y = df['log_income'], 
    alpha=0.5,
)

sns.regplot(data=df, x='education', y='log_income', scatter=False, color='black')
    
plt.title('Effect age has on log_income')
plt.xlabel('Education')
plt.ylabel('Income')
plt.grid(True)

plt.show()

### g.      Analyze a detailed summary of a linear regression model between log_income and education.  What is the R-squared value?  Is the model a good fit?  Is it better than the previous model?

In [ ]:
X = sm.add_constant(df['education'])
Y = df['log_income']

model = sm.OLS(Y, X).fit()

R-квадрат равен 0.541
Да, это очень адекватная модель в контексте одного фактора
Это сильно лучше предыдущей модели

In [ ]:
print(model.summary())
print(f"Критерий Фишера равен {float(model.fvalue):.3f}")

### h.      Analyze a detailed summary of a linear regression model between the dependent variable log_income, and the independent variables age, education, and employment.  Is this model a good fit?  Why?  What conclusions can be made about the different independent variables?

In [ ]:
X = sm.add_constant(df[['education', 'age', 'employment']])
Y = df['log_income']

model = sm.OLS(Y, X).fit()

- Критерий Фишера равен критерию Фишера модели, состоящей только из education
- R-квадрат вырос незначительно
- Смотря на график emploument/income, можно сказать что там не совсем линейная зависимость, но что-то похожее на неё, (Проведя эксперимент путем удаления фактора возраста, критерий Фишера возрос, что говорит о том, что фактор возраста мешает найти зависимость)
- Хотя R-квадрат и растет, это нормально для добавления дополнительных факторов (рост на 0-2 процента), тк просто регрессии становится легче подобрать "подходящую для данной выборки" кривую, но это никак не поможет в целях повышения обобщающей способности

Хотя fit и вышел хорошим статистически, я считаю, что в сравнении с моделью education/income или education+employment/income модель неудачна

In [ ]:
print(model.summary())
print(f"Критерий Фишера равен {float(model.fvalue):.3f}")

In [ ]:
plt.figure(figsize=(8, 5))

plt.scatter(
    x = df['employment'], 
    y = df['log_income'], 
    alpha=0.5,
)

sns.regplot(data=df, x='employment', y='log_income', scatter=False, color='black')
    
plt.title('Effect age has on log_income')
plt.xlabel('Education')
plt.ylabel('Income')
plt.grid(True)

plt.show()

### i.        Based on the coefficients of the multiple regression model, by what percentage would income increase/decrease for every unit of education completed, while all other independent variables remained constant?

In [ ]:
print(model.params)

$$log(income) = 3.5065 + 0.0921ceducation - 0.0027 age + 0.0659 employment$$
$$\Large income = e^{3.5065 + 0.0921 education - 0.0027 age + 0.0659 employment}$$
$$\large income_{new} = e^{0.0921} * e^{3.5065 + 0.0921 education - 0.0027 age + 0.0659 employment}$$
$$\Large income_{new} = income * e^{0.0921}$$

In [ ]:
print(np.e**0.0921)

$$\Large income_{new} = income * 1.0964$$

$$ \Large Следовательно \ увеличивается \ на \ 9.6\text{\%} $$ 

### j.        Create a graph that contains a y = x line and uses the multiple regression model to plot the predicted data points against the actual data points of the training set. 

In [ ]:
Y_real = df['log_income']
Y_pred = model.predict()

plt.figure(figsize=(8, 6))

plt.scatter(Y_real, Y_pred)

min_val = min(Y_real.min(), Y_pred.min())
max_val = max(Y_real.max(), Y_pred.max())
plt.plot([min_val, max_val], [min_val, max_val], color='black')

sns.regplot(x=Y_real, y=Y_pred, scatter=False, color='red')

plt.title('Predicted / actual')
plt.xlabel('Actual')
plt.ylabel('Predicted')
plt.grid(True)

plt.tight_layout()
plt.show()

### k.       How well does the model predict across the various income ranges?

При доходах ниже 4.64 завышает доход
При доходах выше 4.64 занижает доход